In [ ]:
import torch, sys, os
print('PYTHON', sys.version)
print('CUDA', torch.cuda.is_available())
print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('FILES', os.listdir('/content'))

In [ ]:
import zipfile, pathlib
zipfile.ZipFile('/content/overnight_payload.zip').extractall('/content/port_run')
import pandas as pd
P = pd.read_csv('/content/port_run/ft_pairs.csv')
print('DATA_READY', len(P), P.groupby(['src','fold']).size().to_dict())
%pip -q install transformers==4.57.6 peft==0.17.1 bitsandbytes==0.50.2 accelerate rapidfuzz lightgbm
%pip -q uninstall -y torchao
print('DEPENDENCIES_READY')

In [ ]:
import os, gc, time, json, math
from pathlib import Path
import numpy as np, pandas as pd, torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
ROOT=Path('/content/port_run'); ROOT.mkdir(exist_ok=True)
P=pd.read_csv(ROOT/'ft_pairs.csv'); N=pd.read_csv(ROOT/'gen_names.csv')
N=N[N.kor.isin(set(P.kor))].drop_duplicates('kor').copy()
N['fold']=N.kor.map(P.drop_duplicates('kor').set_index('kor').fold).astype(int)
MID='Qwen/Qwen2.5-3B-Instruct'
SYS='Convert a Korean (Hangul) transliteration of a ship name back to the original English/Latin ship name used in international ship registries. Answer with the name only.'
tk=AutoTokenizer.from_pretrained(MID,padding_side='left')
def prompt(kor):
    return tk.apply_chat_template([{'role':'system','content':SYS},{'role':'user','content':str(kor)}],add_generation_prompt=True,tokenize=False)
def status(phase,**kwargs):
    data={'stage':'3b','phase':phase,'time':time.time(),**kwargs}
    (ROOT/'status.json').write_text(json.dumps(data))
    print('STATUS',data,flush=True)
def train3(df):
    torch.manual_seed(0); rng=np.random.RandomState(0)
    m=AutoModelForCausalLM.from_pretrained(MID,torch_dtype=torch.bfloat16).cuda()
    m.config.use_cache=False
    m=get_peft_model(m,LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
    ex=[]
    for kor,eng in zip(df.kor,df.eng):
        a=tk(prompt(kor))['input_ids']; b=tk(str(eng)+'<|im_end|>')['input_ids']; ex.append((a+b,[-100]*len(a)+b))
    bs=16; opt=torch.optim.AdamW([v for v in m.parameters() if v.requires_grad],lr=2e-4)
    steps=2*(len(ex)//bs); sch=torch.optim.lr_scheduler.LambdaLR(opt,lambda s:max(0.,1-s/steps))
    m.train(); start=time.time()
    for ep in range(2):
        order=rng.permutation(len(ex))
        for step,i in enumerate(range(0,len(order)-bs+1,bs)):
            batch=[ex[j] for j in order[i:i+bs]]; L=max(len(a) for a,b in batch)
            ids=torch.tensor([a+[tk.pad_token_id]*(L-len(a)) for a,b in batch],device='cuda')
            labels=torch.tensor([b+[-100]*(L-len(b)) for a,b in batch],device='cuda')
            mask=(torch.arange(L)[None]<torch.tensor([len(a) for a,b in batch])[:,None]).long().cuda()
            loss=m(input_ids=ids,attention_mask=mask,labels=labels).loss
            loss.backward(); opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
            if step%50==0:print('TRAIN3',ep,step,round(loss.item(),4),'seconds',round(time.time()-start),flush=True)
        print('EPOCH3',ep,round(loss.item(),4),round(time.time()-start),flush=True)
    del opt,sch,ids,labels,mask,loss
    m=m.merge_and_unload().eval(); m.config.use_cache=True
    return m
@torch.no_grad()
def generate3(m,names,output):
    done=set(pd.read_csv(output).kor) if output.exists() else set()
    names=[str(s) for s in names if str(s) not in done]
    for start in range(0,len(names),32):
        ns=names[start:start+32]; enc=tk([prompt(s) for s in ns],return_tensors='pt',padding=True).to('cuda')
        g=m.generate(**enc,max_new_tokens=24,num_beams=4,num_return_sequences=4,do_sample=False)
        text=[s.strip().split('\n')[0] for s in tk.batch_decode(g[:,enc['input_ids'].shape[1]:],skip_special_tokens=True)]
        pd.DataFrame([[name]+text[j*4:j*4+4] for j,name in enumerate(ns)],columns=['kor','g1','g2','g3','g4']).to_csv(output,mode='a',header=not output.exists(),index=False)
        if start%320==0:print('GENERATE3',output.name,start,len(names),flush=True)
        del enc,g
for f in [0,1]:
    out=ROOT/f'back3_fold{f}.csv'; modeldir=ROOT/f'model3_fold{f}'
    expected=set(N[N.fold==f].kor)
    if out.exists() and expected.issubset(set(pd.read_csv(out).kor)):
        print('SKIP_COMPLETE_FOLD',f,flush=True); continue
    status('fold_training',fold=f,rows=int((P.fold!=f).sum()))
    if (modeldir/'READY').exists():m=AutoModelForCausalLM.from_pretrained(modeldir,torch_dtype=torch.bfloat16).cuda().eval()
    else:
        m=train3(P[P.fold!=f]); m.save_pretrained(modeldir); tk.save_pretrained(modeldir); (modeldir/'READY').write_text('ok')
    status('fold_generation',fold=f,names=len(expected))
    generate3(m,N[N.fold==f].kor.tolist(),out)
    del m;gc.collect();torch.cuda.empty_cache()
pd.concat([pd.read_csv(ROOT/f'back3_fold{f}.csv') for f in [0,1]],ignore_index=True).to_csv(ROOT/'llm_back_ft_recovered.csv',index=False)
status('full_training',rows=len(P))
modeldir=ROOT/'qwen3b_ft_bf16'
if not (modeldir/'READY').exists():
    m=train3(P);m.save_pretrained(modeldir);tk.save_pretrained(modeldir);(modeldir/'READY').write_text('ok');del m;gc.collect();torch.cuda.empty_cache()
status('quantizing')
Q4=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
q=AutoModelForCausalLM.from_pretrained(modeldir,quantization_config=Q4,torch_dtype=torch.bfloat16,device_map={'':0})
q.save_pretrained(ROOT/'qwen3b_ft_q4');tk.save_pretrained(ROOT/'qwen3b_ft_q4');del q;gc.collect();torch.cuda.empty_cache()
status('DONE',names=len(pd.read_csv(ROOT/'llm_back_ft_recovered.csv')))
print('THREE_B_COMPLETE',flush=True)

In [ ]:
# Runs after the 3B cell. Original 7B QLoRA plus beam-2 comparison.
from peft import prepare_model_for_kbit_training, PeftModel
MID7='Qwen/Qwen2.5-7B-Instruct'
tk7=AutoTokenizer.from_pretrained(MID7,padding_side='left')
Q47=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
def prompt7(kor):
    return tk7.apply_chat_template([{'role':'system','content':SYS},{'role':'user','content':str(kor)}],add_generation_prompt=True,tokenize=False)
def status7(phase,**kwargs):
    data={'stage':'7b','phase':phase,'time':time.time(),**kwargs};(ROOT/'status.json').write_text(json.dumps(data));print('STATUS',data,flush=True)
def base7():
    return AutoModelForCausalLM.from_pretrained(MID7,quantization_config=Q47,torch_dtype=torch.bfloat16,device_map={'':0})
def train7(df):
    torch.manual_seed(0);rng=np.random.RandomState(0)
    m=prepare_model_for_kbit_training(base7(),use_gradient_checkpointing=True)
    m=get_peft_model(m,LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
    ex=[]
    for kor,eng in zip(df.kor,df.eng):
        a=tk7(prompt7(kor))['input_ids'];b=tk7(str(eng)+'<|im_end|>')['input_ids'];ex.append((a+b,[-100]*len(a)+b))
    bs=16;opt=torch.optim.AdamW([v for v in m.parameters() if v.requires_grad],lr=2e-4)
    steps=2*(len(ex)//bs);sch=torch.optim.lr_scheduler.LambdaLR(opt,lambda s:max(0.,1-s/steps));m.train();start=time.time()
    for ep in range(2):
        order=rng.permutation(len(ex))
        for step,i in enumerate(range(0,len(order)-bs+1,bs)):
            batch=[ex[j] for j in order[i:i+bs]];L=max(len(a) for a,b in batch)
            ids=torch.tensor([a+[tk7.pad_token_id]*(L-len(a)) for a,b in batch],device='cuda')
            labels=torch.tensor([b+[-100]*(L-len(b)) for a,b in batch],device='cuda')
            mask=(torch.arange(L)[None]<torch.tensor([len(a) for a,b in batch])[:,None]).long().cuda()
            loss=m(input_ids=ids,attention_mask=mask,labels=labels).loss
            loss.backward();opt.step();sch.step();opt.zero_grad(set_to_none=True)
            if step%50==0:print('TRAIN7',ep,step,round(loss.item(),4),'seconds',round(time.time()-start),flush=True)
        print('EPOCH7',ep,round(loss.item(),4),round(time.time()-start),flush=True)
    del opt,sch,ids,labels,mask,loss
    m.eval();m.config.use_cache=True;m.gradient_checkpointing_disable();return m
@torch.no_grad()
def generate7(m,names,output,beams=1):
    done=set(pd.read_csv(output).kor) if output.exists() else set();names=[str(s) for s in names if str(s) not in done];bs=32 if beams==1 else 16
    for start in range(0,len(names),bs):
        ns=names[start:start+bs];enc=tk7([prompt7(s) for s in ns],return_tensors='pt',padding=True).to('cuda')
        g=m.generate(**enc,max_new_tokens=24,num_beams=beams,num_return_sequences=beams,do_sample=False)
        text=[s.strip().split('\n')[0] for s in tk7.batch_decode(g[:,enc['input_ids'].shape[1]:],skip_special_tokens=True)]
        pd.DataFrame([[name]+text[j*beams:(j+1)*beams] for j,name in enumerate(ns)],columns=['kor']+[f'g{i+1}' for i in range(beams)]).to_csv(output,mode='a',header=not output.exists(),index=False)
        if start%320==0:print('GENERATE7',beams,output.name,start,len(names),flush=True)
        del enc,g
for f in [0,1]:
    ad=ROOT/f'adapter7_fold{f}';expected=set(N[N.fold==f].kor)
    outs=[ROOT/f'back7_greedy_fold{f}.csv',ROOT/f'back7_beam2_fold{f}.csv']
    if all(p.exists() and expected.issubset(set(pd.read_csv(p).kor)) for p in outs):continue
    status7('fold_training',fold=f,rows=int((P.fold!=f).sum()))
    if (ad/'READY').exists():m=PeftModel.from_pretrained(base7(),ad).eval()
    else:
        m=train7(P[P.fold!=f]);m.save_pretrained(ad);(ad/'READY').write_text('ok')
    status7('fold_generation',fold=f,names=len(expected))
    generate7(m,N[N.fold==f].kor.tolist(),outs[0],1)
    generate7(m,N[N.fold==f].kor.tolist(),outs[1],2)
    del m;gc.collect();torch.cuda.empty_cache()
for mode in ['greedy','beam2']:
    pd.concat([pd.read_csv(ROOT/f'back7_{mode}_fold{f}.csv') for f in [0,1]],ignore_index=True).to_csv(ROOT/f'llm_back_q7_{mode}_recovered.csv',index=False)
ad=ROOT/'q7_adapter_full';status7('full_training',rows=len(P))
if not (ad/'READY').exists():
    m=train7(P);m.save_pretrained(ad);(ad/'READY').write_text('ok');del m;gc.collect();torch.cuda.empty_cache()
bf=ROOT/'q7_merged_bf16';status7('merge')
if not (bf/'READY').exists():
    base=AutoModelForCausalLM.from_pretrained(MID7,torch_dtype=torch.bfloat16,device_map={'':0})
    merged=PeftModel.from_pretrained(base,ad).merge_and_unload();merged.save_pretrained(bf,max_shard_size='4GB');tk7.save_pretrained(bf);(bf/'READY').write_text('ok')
    del base,merged;gc.collect();torch.cuda.empty_cache()
status7('quantizing')
q=AutoModelForCausalLM.from_pretrained(bf,quantization_config=Q47,torch_dtype=torch.bfloat16,device_map={'':0})
q.save_pretrained(ROOT/'qwen7b_ft_q4');tk7.save_pretrained(ROOT/'qwen7b_ft_q4');del q;gc.collect();torch.cuda.empty_cache()
status7('DONE',names=len(N));print('SEVEN_B_COMPLETE',flush=True)

In [ ]:
import os, pathlib
p=pathlib.Path('/content/port_run')
print('RECOVERY_FILES',p.exists(),[x.name for x in p.glob('*')])


In [ ]:
%pip -q install transformers==4.57.6 huggingface-hub==0.36.2 tokenizers==0.22.1 lightgbm rapidfuzz accelerate
import torch, pathlib, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
p=pathlib.Path('/content/qwen3b');p.mkdir(exist_ok=True)
m=AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct',torch_dtype=torch.bfloat16)
m.save_pretrained(p);AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct').save_pretrained(p)
del m;gc.collect();print('GPU_REHEARSAL_WEIGHTS_READY',torch.cuda.is_available(),flush=True)


In [ ]:
import sys,zipfile,pathlib,shutil,json,subprocess,time,pandas as pd,joblib,os
R=pathlib.Path('/content/ninecheck');R.mkdir(exist_ok=True)
zipfile.ZipFile('/content/nine_gpu_payload.zip').extractall(R)
sys.path.insert(0,str(R/'nine/R01/model'))
for f in (R/'original/model/qwen3b').glob('*'):
    if f.is_file():shutil.copyfile(f,pathlib.Path('/content/qwen3b')/f.name)
t=pd.read_csv(R/'data/train.csv');k=t.vessel_name.str.contains('[가-힣]',na=False)
big=pd.concat([t[k].sample(100,random_state=31),t[~k].sample(200,random_state=31)]).drop(columns='target_vessel_id')
small=pd.concat([t[k].sample(6,random_state=31),t[~k].sample(12,random_state=31)]).drop(columns='target_vessel_id')
results=[]
for number in [3,6,9,1,2,4,5,7,8]:
    kor=['robust','robust_deep','control'][(number-1)//3];en=['balanced','name_focus','regularized'][(number-1)%3]
    folder=R/f'run{number:02d}';shutil.copytree(R/'nine/R01',folder,dirs_exist_ok=True)
    md=folder/'model'
    if not (md/'qwen3b').exists():os.symlink('/content/qwen3b',md/'qwen3b',target_is_directory=True)
    shutil.copyfile(R/f'variants/en_{en}.txt',md/'en_ranker.txt');shutil.copyfile(R/f'variants/strict_{kor}.txt',md/f'strict_{kor}.txt')
    meta=joblib.load(R/'nine/R01/model/meta.pkl')
    if en=='name_focus':meta['en_fs']=[x for x in meta['en_fs'] if x not in ('pflag','ptyp','tgrp')]
    joblib.dump(meta,md/'meta.pkl');(md/'choice.json').write_text(json.dumps({'name':kor}))
    (folder/'data').mkdir(exist_ok=True);shutil.copyfile(R/'data/vessel_catalog.csv',folder/'data/vessel_catalog.csv')
    sample=big if number==3 else small;sample.to_csv(folder/'data/test.csv',index=False)
    start=time.time();p=subprocess.run(['python','script.py'],cwd=folder,capture_output=True,text=True,timeout=240)
    print('GPU_RUN',number,p.returncode,round(time.time()-start),p.stdout[-400:],p.stderr[-900:],flush=True)
    if p.returncode:raise RuntimeError(f'GPU candidate {number} failed')
    s=pd.read_csv(folder/'output/submission.csv');assert len(s)==len(sample) and s.target_vessel_id.notna().all()
    results.append({'candidate':number,'rows':len(s),'seconds':round(time.time()-start)})
    (R/'gpu_results.json').write_text(json.dumps(results))
print('ALL_NINE_GPU_PASSED',results,flush=True)
